# Impact Fund Name Screener

Identifies funds whose name suggests an impact mandate via regex matching against Dirk's keyword list (English + multilingual equivalents).

**Run:** Kernel → Restart & Run All  
**Edit:** Cell 1 (paths/columns) and Cell 2 (patterns) only.

## CELL 1 — Configuration

Edit paths and column list here. All other cells run without changes.

In [ ]:
import os, re, json
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
  
config = {}
with open("File_Directory.txt") as f:
    for line in f:
        if ":" in line:
            key, val = line.split(":", 1)
            config[key.strip()] = val.strip()

INPUT_FILE  = Path(config["Input"])
OUTPUT_DIR  = Path(config["Output"])

NAME_COL    = "Name"
ID_COL      = "FundId"

OBJECTIVE_COLUMNS = [
    "Prospectus Objective",
    "KIID Objective/Investment Policy",
    "PRIIPS KID Objective",
    "Strategy Description",
    "Investment Strategy - English",
    "PRIIPS KID Objective - Danish",
    "PRIIPS KID Objective - Dutch",
    "PRIIPS KID Objective - Finnish",
    "PRIIPS KID Objective - French",
    "PRIIPS KID Objective - German",
    "PRIIPS KID Objective - Italian",
    "PRIIPS KID Objective - Norwegian",
    "PRIIPS KID Objective - Portuguese",
    "PRIIPS KID Objective - Spanish",
    "PRIIPS KID Objective - Swedish",
    "KIID Objective/Investment Policy - German",
    "KIID Objective/Investment Policy - French",
    "KIID Objective/Investment Policy - Italian",
    "KIID Objective/Investment Policy - Spanish",
    "KIID Objective/Investment Policy - Norwegian",
    "KIID Objective/Investment Policy - Swedish",
    "KIID Objective/Investment Policy - Finnish",
    "KIID Objective/Investment Policy - Portuguese",
    "KIID Objective/Investment Policy - Danish",
    "Investment Strategy - Danish",
    "Investment Strategy - Finnish",
    "Investment Strategy - French",
    "Investment Strategy - German",
    "Investment Strategy - Italian",
    "Investment Strategy - Norwegian",
    "Investment Strategy - Portuguese",
    "Investment Strategy - Spanish",
    "Investment Strategy - Swedish",
]

print("Configuration loaded.")
print(f"  Input:  {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")


Configuration loaded.
  Input:  /Users/dannyhogan/Desktop/Hogan_RA_Work/Download Sustainable Funds 2026-04-15.xlsx
  Output: /Users/dannyhogan/Desktop/Hogan_RA_Work


## CELL 2 — Keyword Patterns

One entry per `(label, language, regex, needs_review)`. `needs_review=True` flags the match for human inspection — use for ambiguous abbreviations or high false-positive-risk terms.

Merged against `RegexTerms_edited.xlsx` ("Impact terms by language") — every one of the 237 non-empty term cells in that sheet is verified to match at least one pattern below (checked programmatically, not by eye). This added 3 new concepts (Green change, Sustainable change, Social change), 1 new concept cluster (Active ownership), and ~30 new per-language entries for existing concepts, plus fixed a few that were silently broken: the shared EN/FR/DE/SV/DA "transformative" suffix pattern only actually matched the English spelling; German transition was assumed to route through the English loanword or "Energiewende" but the xlsx's real term is bare "Wende"; and the Swedish SDG regex had an `en?` typo that made the trailing "en" suffix non-optional (fixed to `(?:en)?`).

In [ ]:
# Each entry: (label, language, regex_pattern, needs_review)
#
# needs_review=True  → match is flagged for human inspection
#                       because the abbreviation is ambiguous
#                       or the term has false-positive risk.
# ============================================================
# Merged against RegexTerms_edited.xlsx ("Impact terms by language") to be
# exhaustive against every cell in that sheet, including alternate/secondary
# terms listed under a primary term. New entries below are marked "# NEW".
# needs_review on NEW entries reflects genuine ambiguity/false-positive risk
# (assessed per-term), not "this pattern is new" — see the existing
# 'transition'/EN-FR-DE entry for why those two things are not the same.
# ============================================================

# re.IGNORECASE applied at compile time for all patterns
FLAGS = re.IGNORECASE

PATTERNS = [
 
    # ===== IMPACT =====
    ('impact'                  , 'EN/FR/DE/NL/SV/DA/NO'    , 'impact'                      , r"\bimpact\b", False),  # 57 funds
    ('impact'                  , 'EN'                      , 'impact_num'                  , r"\bimpact\d", True),  # NEW 3 funds — Impact360; \bimpact\b misses digit suffix
    ('impact'                  , 'ES/PT'                   , 'impacto'                     , r"\bimpacto\b", False),  # 2 funds
    ('impact'                  , 'IT'                      , 'impatto'                     , r"\bimpatto\b", False),  # 0 funds
    ('impact'                  , 'FI'                      , 'vaikuttavuus'                , r"\bvaikuttavuu|\bvaikuttava", False),  # 0 funds
    ('impact'                  , 'DE'                      , 'wirkung'                     , r"\bwirkung\b", True),  # NEW — xlsx alt; generic German word for "effect/impact", high FP risk
    ('impact'                  , 'SV'                      , 'paverkan'                    , r"\bp[åa]verkan\b", True),  # NEW — xlsx alt; generic Swedish "impact/influence", high FP risk
    ('impact'                  , 'DA'                      , 'pavirkning_da'               , r"\bp[åa]virkning\b", True),  # NEW — xlsx alt; generic Danish "impact/influence", high FP risk
    ('impact'                  , 'NO'                      , 'pavirke_no'                  , r"\bp[åa]virke\b|\bp[åa]virkning\b", True),  # NEW — xlsx alt; generic Norwegian "impact/influence" (verb+noun), high FP risk
 
    # ===== POSITIVE CHANGE =====
    ('positive change'         , 'EN'                      , 'positive_change'             , r"\bpositive[\s\-]change\b", False),  # 2 funds
    ('positive change'         , 'EN'                      , '4change'                     , r"\b4[Cc]hange\b", False),  # 3 funds — brand (R-co 4Change)
    ('positive change'         , 'FR'                      , 'changement_positif'          , r"\bchangement[\s\-]positif\b", False),  # 0 funds
    ('positive change'         , 'DE'                      , 'positiver_wandel'            , r"\bpositiver?[\s\-]wandel\b", False),  # 0 funds
    ('positive change'         , 'DE'                      , 'positiver_beitrag'           , r"\bpositiver?[\s\-]beitrag\b", False),  # NEW — xlsx alt "Positiver Beitrag" (positive contribution)
    ('positive change'         , 'ES'                      , 'cambio_positivo'             , r"\bcambio[\s\-]positivo\b", False),  # 0 funds
    ('positive change'         , 'IT'                      , 'cambiamento_positivo'        , r"\bcambiamento[\s\-]positivo\b", False),  # 0 funds
    ('positive change'         , 'PT'                      , 'mudanca_positiva'            , r"\bmudan[çc]a[\s\-]positiva\b", False),  # 0 funds
    ('positive change'         , 'NL'                      , 'positieve_verandering'       , r"\bpositieve[\s\-]verandering\b", False),  # 0 funds
    ('positive change'         , 'SV'                      , 'positiv_forandring_sv'       , r"\bpositiv[\s\-]f[öo]r[äa]ndring\b", False),  # 0 funds
    ('positive change'         , 'DA'                      , 'positiv_forandring_da'       , r"\bpositiv[\s\-]forandring\b", False),  # 0 funds
    ('positive change'         , 'DA'                      , 'positiv_udvikling'           , r"\bpositiv[\s\-]udvikling\b", False),  # NEW — xlsx alt "Positiv udvikling" (positive development)
    ('positive change'         , 'NO'                      , 'positiv_endring_no'          , r"\bpositiv[\s\-](?:endring|forandring)\b", False),  # 0 funds
    ('positive change'         , 'NO'                      , 'positiv_utvikling'           , r"\bpositiv[\s\-]utvikling\b", False),  # NEW — xlsx alt "Positiv utvikling" (positive development)
    ('positive change'         , 'FI'                      , 'myonteinen_muutos'           , r"\b(?:positiivinen|myönteinen)[\s\-]muutos\b", False),  # 0 funds
 
    # ===== GREEN CHANGE ===== (NEW concept, from xlsx)
    ('green change'            , 'EN'                      , 'green_change'                , r"\bgreen[\s\-]change\b", False),  # NEW
    ('green change'            , 'FR'                      , 'changement_vert'             , r"\bchangement[\s\-]vert\b", False),  # NEW
    ('green change'            , 'DE'                      , 'gruner_wandel'               , r"\bgr[üu]ner[\s\-]wandel\b", False),  # NEW
    ('green change'            , 'ES'                      , 'cambio_verde'                , r"\bcambio[\s\-]verde\b", False),  # NEW
    ('green change'            , 'IT'                      , 'cambiamento_verde'           , r"\bcambiamento[\s\-]verde\b", False),  # NEW
    ('green change'            , 'PT'                      , 'mudanca_verde'               , r"\bmudan[çc]a[\s\-]verde\b", False),  # NEW
    ('green change'            , 'NL'                      , 'groene_verandering'          , r"\bgroene[\s\-]verandering\b", False),  # NEW
    ('green change'            , 'SV'                      , 'gron_forandring_sv'          , r"\bgr[öo]n[\s\-]f[öo]r[äa]ndring\b", False),  # NEW
    ('green change'            , 'DA'                      , 'gron_forandring_da'          , r"\bgr[øo]n[\s\-]forandring\b", False),  # NEW
    ('green change'            , 'NO'                      , 'gronn_endring'               , r"\bgr[øo]nn[\s\-]endring\b", False),  # NEW
    ('green change'            , 'FI'                      , 'vihrea_muutos'               , r"\bvihre[äa][\s\-]muutos\b", False),  # NEW
 
    # ===== SUSTAINABLE CHANGE ===== (NEW concept, from xlsx)
    ('sustainable change'      , 'EN'                      , 'sustainable_change'          , r"\bsustainable[\s\-]change\b", False),  # NEW
    ('sustainable change'      , 'FR'                      , 'changement_durable'          , r"\bchangement[\s\-]durable\b", False),  # NEW
    ('sustainable change'      , 'DE'                      , 'nachhaltiger_wandel'         , r"\bnachhaltiger?[\s\-]wandel\b", False),  # NEW
    ('sustainable change'      , 'ES'                      , 'cambio_sostenible'           , r"\bcambio[\s\-]sostenible\b", False),  # NEW
    ('sustainable change'      , 'IT'                      , 'cambiamento_sostenibile'     , r"\bcambiamento[\s\-]sostenibile\b", False),  # NEW
    ('sustainable change'      , 'PT'                      , 'mudanca_sustentavel'         , r"\bmudan[çc]a[\s\-]sustent[áa]vel\b", False),  # NEW
    ('sustainable change'      , 'NL'                      , 'duurzame_verandering'        , r"\bduurzame[\s\-]verandering\b", False),  # NEW
    ('sustainable change'      , 'SV'                      , 'hallbar_forandring'          , r"\bh[åä]llbar[\s\-]f[öo]r[äa]ndring\b", False),  # NEW
    ('sustainable change'      , 'DA'                      , 'baeredygtig_forandring'      , r"\bb[æa]redygtig[\s\-]forandring\b", False),  # NEW
    ('sustainable change'      , 'NO'                      , 'baerekraftig_endring'        , r"\bb[æa]rekraftig[\s\-]endring\b", False),  # NEW
    ('sustainable change'      , 'FI'                      , 'kestava_muutos'              , r"\bkest[äa]v[äa][\s\-]muutos\b", False),  # NEW
 
    # ===== SOCIAL CHANGE ===== (NEW concept, from xlsx)
    ('social change'           , 'EN'                      , 'social_change'               , r"\bsocial[\s\-]change\b", False),  # NEW
    ('social change'           , 'FR'                      , 'changement_social'           , r"\bchangement[\s\-]social\b", False),  # NEW
    ('social change'           , 'DE'                      , 'sozialer_wandel'             , r"\bsozialer?[\s\-]wandel\b", False),  # NEW
    ('social change'           , 'ES'                      , 'cambio_social'               , r"\bcambio[\s\-]social\b", False),  # NEW
    ('social change'           , 'IT'                      , 'cambiamento_sociale'         , r"\bcambiamento[\s\-]sociale\b", False),  # NEW
    ('social change'           , 'PT'                      , 'mudanca_social'              , r"\bmudan[çc]a[\s\-]social\b", False),  # NEW
    ('social change'           , 'NL'                      , 'sociale_verandering'         , r"\bsociale[\s\-]verandering\b", False),  # NEW
    ('social change'           , 'SV'                      , 'social_forandring_sv'        , r"\bsocial[\s\-]f[öo]r[äa]ndring\b", False),  # NEW
    ('social change'           , 'DA'                      , 'social_forandring_da'        , r"\bsocial[\s\-]forandring\b", False),  # NEW
    ('social change'           , 'NO'                      , 'sosial_endring'              , r"\bsosial[\s\-]endring\b", False),  # NEW
    ('social change'           , 'FI'                      , 'sosiaalinen_muutos'          , r"\bsosiaalinen[\s\-]muutos\b", False),  # NEW
 
    # ===== BETTER WORLD =====
    ('better world'            , 'EN'                      , 'better_world'                , r"\bbetter[\s\-]world\b", False),  # 3 funds
    ('better world'            , 'EN'                      , 'better_future'               , r"\bbetter[\s\-]future\b", False),  # 1 funds
    ('better world'            , 'FR'                      , 'meilleur_monde'              , r"\bmeilleur[\s\-]monde\b|\bmonde[\s\-]meilleur\b", False),  # 0 funds
    ('better world'            , 'DE'                      , 'bessere_welt'                , r"\bbessere[\s\-]welt\b", False),  # 0 funds
    ('better world'            , 'ES'                      , 'mejor_mundo'                 , r"\bmejor[\s\-]mundo\b|\bmundo[\s\-]mejor\b", False),  # 0 funds
    ('better world'            , 'IT'                      , 'mondo_migliore'              , r"\bmondo[\s\-]migliore\b", False),  # 0 funds
    ('better world'            , 'PT'                      , 'mundo_melhor'                , r"\bmundo[\s\-]melhor\b", False),  # 0 funds
    ('better world'            , 'NL'                      , 'betere_wereld'               , r"\bbetere[\s\-]wereld\b", False),  # 0 funds
    ('better world'            , 'SV'                      , 'battre_varld'                , r"\bb[äa]ttre[\s\-]v[äa]rld\b", False),  # 0 funds
    ('better world'            , 'SV'                      , 'varldsforbattrande'          , r"\bv[äa]rldsf[öo]rb[äa]ttrande\b", False),  # NEW — xlsx alt "Världsförbättrande" (compound, world-improving)
    ('better world'            , 'DA/NO'                   , 'bedre_verden'                , r"\bbedre[\s\-]verden\b", False),  # 0 funds
    ('better world'            , 'FI'                      , 'parempi_maailma'             , r"\bparempi[\s\-]maailma\b", False),  # 0 funds
 
    # ===== FUTURE GENERATIONS =====
    ('future generations'      , 'EN'                      , 'future_generations'          , r"\bfuture[\s\-]generations?\b", False),  # 1 funds
    ('future generations'      , 'FR'                      , 'generations_futures'         , r"\bg[ée]n[ée]rations?[\s\-]futures?\b", False),  # 1 funds
    ('future generations'      , 'DE'                      , 'kunftige_generationen'       , r"\b(?:zu)?k[üu]nftige[\s\-]generationen\b", False),  # 0 funds
    ('future generations'      , 'DE'                      , 'kommende_generationen'       , r"\bkommende[\s\-]generationen\b", False),  # NEW — xlsx alt "Kommende Generationen"; different stem, not covered by (zu)?kunftige
    ('future generations'      , 'ES'                      , 'generaciones_futuras'        , r"\bgeneraciones?[\s\-]futuras?\b", False),  # 0 funds
    ('future generations'      , 'IT'                      , 'generazioni_future'          , r"\bgenerazioni[\s\-]future\b", False),  # 0 funds
    ('future generations'      , 'PT'                      , 'geracoes_futuras'            , r"\bgera[çc][õo]es[\s\-]futuras\b", False),  # 0 funds
    ('future generations'      , 'NL'                      , 'toekomstige_generaties'      , r"\btoekomstige[\s\-]generaties\b", False),  # 0 funds
    ('future generations'      , 'SV'                      , 'framtida_generationer'       , r"\bframtida[\s\-]generationer\b", False),  # 0 funds
    ('future generations'      , 'DA'                      , 'fremtidige_generationer_da'  , r"\bfremtidige[\s\-]generationer\b", False),  # 0 funds
    ('future generations'      , 'NO'                      , 'fremtidige_generasjoner'     , r"\bfremtidige[\s\-]generasjoner\b", False),  # 0 funds
    ('future generations'      , 'FI'                      , 'tulevat_sukupolvet'          , r"\btulev\w+[\s\-]sukupolv", False),  # 0 funds — also covers xlsx alt "Tuleville sukupolville"; xlsx literally has "Ttuleville" (double-T), looks like a typo, not special-cased
 
    # ===== FUTURE FOR GENERATIONS =====
    ('future for generations'  , 'EN'                      , 'for_generations'             , r"\bfor[\s\-]gen(?:eration|)s?\b", True),  # NEW 3 funds — Food/Future For Generations/Gens
    ('future for generations'  , 'FR'                      , 'pour_generations'            , r"\bpour[\s\-](?:les[\s\-])?g[ée]n[ée]rations\b", False),  # 0 funds
    ('future for generations'  , 'DE'                      , 'fur_generationen'            , r"\bf[üu]r[\s\-](?:die[\s\-])?generationen\b", False),  # 0 funds
    ('future for generations'  , 'ES'                      , 'para_generaciones'           , r"\bpara[\s\-](?:las[\s\-])?generaciones\b", False),  # 0 funds
    ('future for generations'  , 'IT'                      , 'per_generazioni'             , r"\bper[\s\-](?:le[\s\-])?generazioni\b", False),  # 0 funds
    ('future for generations'  , 'PT'                      , 'para_geracoes'               , r"\bpara[\s\-](?:as[\s\-])?gera[çc][õo]es\b", False),  # 0 funds
    ('future for generations'  , 'NL'                      , 'voor_generaties'             , r"\bvoor[\s\-](?:de[\s\-])?generaties\b", False),  # 0 funds
    ('future for generations'  , 'SV'                      , 'for_generationer_sv'         , r"\bf[öo]r[\s\-]generationer\b", False),  # 0 funds
    ('future for generations'  , 'DA'                      , 'for_generationer_da'         , r"\bfor[\s\-]generationer\b", False),  # 0 funds
    ('future for generations'  , 'NO'                      , 'for_generasjoner'            , r"\bfor[\s\-]generasjoner\b", False),  # 0 funds
    ('future for generations'  , 'FI'                      , 'sukupolville'                , r"\bsukupolville\b", False),  # 0 funds
 
    # ===== SUSTAINABLE DEVELOPMENT =====
    ('sustainable development' , 'EN'                      , 'sustainable_development'     , r"\bsustainable[\s\-]dev(?:elopment)?\b", False),  # 0 funds
    ('sustainable development' , 'FR'                      , 'dev_durable'                 , r"\bd[ée]veloppement[\s\-]durable\b", False),  # 2 funds
    ('sustainable development' , 'DE'                      , 'nachhaltige_entwicklung'     , r"\bnachhaltige[\s\-]entwicklung\b", False),  # 0 funds
    ('sustainable development' , 'ES'                      , 'desarrollo_sostenible'       , r"\bdesarrollo[\s\-]sostenible\b", False),  # 0 funds
    ('sustainable development' , 'IT'                      , 'sviluppo_sostenibile'        , r"\bsviluppo[\s\-]sostenibile\b", False),  # 0 funds
    ('sustainable development' , 'PT'                      , 'desenvolvimento_sustentavel' , r"\bdesenvolvimento[\s\-]sustent[áa]vel\b", False),  # 0 funds
    ('sustainable development' , 'NL'                      , 'duurzame_ontwikkeling'       , r"\bduurzame[\s\-]ontwikkeling\b", False),  # 0 funds
    ('sustainable development' , 'SV'                      , 'hallbar_utveckling'          , r"\bh[åä]llbar[\s\-]utveckling\b", False),  # 0 funds
    ('sustainable development' , 'DA'                      , 'baeredygtig_udvikling'       , r"\bb[æa]redygtig[\s\-]udvikling\b", False),  # 0 funds
    ('sustainable development' , 'NO'                      , 'baerekraftig_utvikling'      , r"\bb[æa]rekraftig[\s\-]utvikling\b", False),  # 0 funds
    ('sustainable development' , 'FI'                      , 'kestava_kehitys'             , r"\bkest[äa]v[äa][\s\-]kehitys\b", False),  # 0 funds
 
    # ===== SDG =====
    ('SDG'                     , 'EN/DE/NL/IT/SV/DA/NO/FI' , 'sdg'                         , r"\bSDGs?\b", False),  # 17 funds — international acronym
    ('SDG'                     , 'EN'                      , 'sust_dev_goals'              , r"\bsust\w*[\s\-]develop\w*[\s\-]goals?\b", True),  # NEW 1 funds — "Sust Developt Goals" spelled out
    ('SDG'                     , 'FR'                      , 'odd'                         , r"\bODD\b", True),  # NEW 2 funds — Objectifs de Developpement Durable (excludes 'Oddo')
    ('SDG'                     , 'DE'                      , 'nachhaltigkeitsziele'        , r"\bnachhaltigkeitsziele\b", False),  # NEW — xlsx: German native term, not just the SDG loanword
    ('SDG'                     , 'ES/PT'                   , 'ods'                         , r"\bODS\b", False),  # 0 funds — Objetivos de Desarrollo Sostenible
    ('SDG'                     , 'IT'                      , 'oss'                         , r"\bOSS\b", True),  # NEW — xlsx: Italian acronym (Obiettivi di Sviluppo Sostenibile); very short/generic, high FP risk
    ('SDG'                     , 'NL'                      , 'duurzame_ontwikkelingsdoelen', r"\bduurzame[\s\-]ontwikkelingsdoelen\b", False),  # NEW — xlsx: Dutch native term
    ('SDG'                     , 'NL'                      , 'werelddoelen'                , r"\bwerelddoelen\b", False),  # NEW — xlsx alt "Werelddoelen" (world goals), Dutch informal term for SDGs
    ('SDG'                     , 'DA'                      , 'verdensmaal'                 , r"\bverdensm[åa]l\b", False),  # 0 funds — Danish 'Verdensmaal'
    ('SDG'                     , 'NO'                      , 'baerekraftsmaal'             , r"\bb[æa]rekraftsm[åa]l\b", False),  # 0 funds — Norwegian 'Baerekraftsmaal'
    ('SDG'                     , 'SV'                      , 'globala_malen'               , r"\bglobala[\s\-]m[åa]l(?:en)?\b", False),  # MODIFIED — "en" suffix now fully optional (was "en?" which only made the 'n' optional, a regex bug); xlsx has both "Globala målen" and "Globala mål"
    ('SDG'                     , 'FI'                      , 'kestavan_kehityksen_tavoitteet', r"\bkest[äa]v[äa]n[\s\-]kehityksen[\s\-]tavoitteet\b", False),  # NEW — xlsx: Finnish full-phrase term
 
    # ===== TRANSFORMATION =====
    ('transformation'          , 'EN/FR/DE/SV/DA'          , 'transformation'              , r"\btransform(?:ation|ative|ations)?\b", False),  # 2 funds — EN also matches 'transformative'
    ('transformation'          , 'FR'                      , 'transformer'                 , r"\btransformer\b", True),  # NEW — xlsx alt "Transformer" (verb); common generic French verb, high FP risk
    ('transformation'          , 'ES'                      , 'transformacion'              , r"\btransformaci[óo]n\b", False),  # 0 funds
    ('transformation'          , 'IT'                      , 'trasformazione'              , r"\btrasformazione\b", False),  # 0 funds
    ('transformation'          , 'PT'                      , 'transformacao'               , r"\btransforma[çc][ãa]o\b", False),  # 0 funds
    ('transformation'          , 'NL'                      , 'transformatie'               , r"\btransformatie\b", False),  # 0 funds
    ('transformation'          , 'NO'                      , 'transformasjon'              , r"\btransformasjon\b", False),  # 0 funds
    ('transformation'          , 'FI'                      , 'transformaatio'              , r"\btransformaatio\b", False),  # 0 funds

    # ── transformative (xlsx lists as its own row; the shared EN/FR/DE/SV/DA
    # regex above only catches the ENGLISH "-ative" spelling — French
    # "Transformateur", German/Swedish/Danish/Norwegian "Transformativ" (no
    # final e) and the ES/IT/PT/NL/FI equivalents all miss it entirely. Added
    # as explicit per-language entries instead of widening the shared regex,
    # since some of these (French, Spanish, Portuguese) are also common
    # generic/homonym words. ──
    ('transformation'          , 'FR'                      , 'transformateur'              , r"\btransformateur\b", True),  # NEW — xlsx alt "Transformateur"; ALSO the ordinary French word for an electrical transformer — high FP risk
    ('transformation'          , 'DE'                      , 'transformativ'               , r"\btransformativ\b", False),  # NEW — xlsx alt "Transformativ"; not matched by shared regex (needs final 'e')
    ('transformation'          , 'ES'                      , 'transformador'               , r"\btransformador\b", True),  # NEW — xlsx alt "Transformador"; ALSO the ordinary Spanish word for an electrical transformer — high FP risk
    ('transformation'          , 'IT'                      , 'trasformativo'               , r"\btrasformativo\b", False),  # NEW — xlsx alt "Trasformativo"
    ('transformation'          , 'PT'                      , 'transformador_pt'            , r"\btransformador\b", True),  # NEW — xlsx alt "Transformador"; ALSO the ordinary Portuguese word for an electrical transformer — high FP risk
    ('transformation'          , 'NL'                      , 'transformatief'              , r"\btransformatief\b", False),  # NEW — xlsx alt "Transformatief"
    ('transformation'          , 'SV'                      , 'transformativ_sv'            , r"\btransformativ\b", False),  # NEW — xlsx alt "Transformativ"; not matched by shared regex
    ('transformation'          , 'DA'                      , 'transformativ_da'            , r"\btransformativ\b", False),  # NEW — xlsx alt "Transformativ"; not matched by shared regex
    ('transformation'          , 'NO'                      , 'transformativ_no'            , r"\btransformativ\b", False),  # NEW — xlsx alt "Transformativ"; NO wasn't even in the shared group
    ('transformation'          , 'FI'                      , 'transformatiivinen'          , r"\btransformatiivinen\b", False),  # NEW — xlsx alt "Transformatiivinen"
 
    # ===== TRANSITION =====
    ('transition'              , 'EN/FR/DE'                , 'transition'                  , r"\btransition\b", False),  # 35 funds — lowest-precision term; matches 'Transition Materials' etc.
    ('transition'              , 'DE'                      , 'energiewende'                , r"\benergie?wende\b", False),  # 2 funds — German energy transition
    ('transition'              , 'DE'                      , 'wende'                       , r"\bwende\b", True),  # NEW — xlsx: bare "Wende" is the canonical DE transition term here, but it's an extremely common, generic German noun ("turn"/"change", historically "die Wende"=German reunification) — very high FP risk
    ('transition'              , 'ES'                      , 'transicion'                  , r"\btransici[óo]n\b", False),  # 1 funds
    ('transition'              , 'IT'                      , 'transizione'                 , r"\btransizione\b", False),  # 0 funds
    ('transition'              , 'PT'                      , 'transicao'                   , r"\btransi[çc][ãa]o\b", False),  # 0 funds
    ('transition'              , 'NL'                      , 'transitie'                   , r"\btransitie\b", False),  # 0 funds
    ('transition'              , 'SV'                      , 'omstallning'                 , r"\bomst[åä]llning\b", False),  # 0 funds
    ('transition'              , 'DA/NO'                   , 'omstilling'                  , r"\bomstilling\b", False),  # 0 funds — can also mean 'restructuring'
    ('transition'              , 'FI'                      , 'siirtyma'                    , r"\bsiirtym[äa]\b", False),  # 0 funds
 
    # ===== CLIMATE ACTION =====
    ('climate action'          , 'EN'                      , 'climate_action'              , r"\bclimate[\s\-]action\b", False),  # 4 funds
    ('climate action'          , 'EN'                      , 'act_for_climate'             , r"\bact[\s\-]for[\s\-]climate\b", True),  # NEW 1 funds — FIDEAS ACT for CLIMATE
    ('climate action'          , 'FR'                      , 'action_climatique'           , r"\baction[\s\-]climatique\b", False),  # 0 funds — full phrase only — bare 'action'='stock' in FR
    ('climate action'          , 'FR'                      , 'contribution_climat'         , r"\bcontribution[\s\-]climat\b", False),  # NEW — xlsx alt "Contribution climat"
    ('climate action'          , 'DE'                      , 'klimaaktion'                 , r"\bklimaaktion\b", False),  # 0 funds
    ('climate action'          , 'ES'                      , 'accion_climatica'            , r"\bacci[óo]n[\s\-]clim[áa]tica\b", False),  # 0 funds
    ('climate action'          , 'IT'                      , 'azione_climatica'            , r"\bazione[\s\-]climatica\b", False),  # 0 funds
    ('climate action'          , 'PT'                      , 'acao_climatica'              , r"\ba[çc][ãa]o[\s\-]clim[áa]tica\b", False),  # 0 funds
    ('climate action'          , 'NL'                      , 'klimaatactie'                , r"\bklimaatactie\b", False),  # 0 funds
    ('climate action'          , 'SV'                      , 'klimataktion'                , r"\bklimat(?:aktion|åtg[äa]rd)\b", False),  # 0 funds
    ('climate action'          , 'DA'                      , 'klimahandling_da'            , r"\bklima(?:handling|indsats)\b", False),  # 0 funds
    ('climate action'          , 'NO'                      , 'klimahandling_no'            , r"\bklima(?:handling|tiltak)\b", False),  # 0 funds
    ('climate action'          , 'FI'                      , 'ilmastotoimet'               , r"\bilmasto(?:toimet?|teko)\b", False),  # 0 funds
    ('climate action'          , 'FI'                      , 'ilmastovaikuttaminen'        , r"\bilmastovaikuttaminen\b", False),  # NEW — xlsx alt "Ilmastovaikuttaminen"
 
    # ===== ENGAGEMENT =====
    ('engagement'              , 'EN/FR/DE/DA/NL'          , 'engagement'                  , r"\bengagement\b", False),  # 5 funds
    ('engagement'              , 'EN'                      , 'engagement_abbrev'           , r"\beng(?:a|mnt|mt)\b", True),  # NEW 4 funds — Enga/Engmnt truncations; excludes full 'engagement'
    ('engagement'              , 'DE'                      , 'unternehmensdialog'          , r"\bunternehmensdialog\b", False),  # NEW — xlsx alt "Unternehmensdialog" (corporate dialogue)
    ('engagement'              , 'ES'                      , 'compromiso'                  , r"\bcompromiso\b", False),  # 1 funds — GENERIC ('commitment') — false-positive risk
    ('engagement'              , 'ES'                      , 'implicacion'                 , r"\bimplicaci[óo]n\b", False),  # NEW — xlsx PRIMARY term for ES; GENERIC ('involvement') — false-positive risk, same treatment as 'compromiso' sibling
    ('engagement'              , 'IT'                      , 'impegno'                     , r"\bimpegno\b", False),  # 0 funds — GENERIC ('commitment') — false-positive risk
    ('engagement'              , 'IT'                      , 'dialogo_societario'          , r"\bdialogo[\s\-]societario\b", False),  # NEW — xlsx PRIMARY term for IT (corporate dialogue), more specific than 'impegno'
    ('engagement'              , 'PT'                      , 'engajamento'                 , r"\bengajamento\b", False),  # 0 funds
    ('engagement'              , 'PT'                      , 'envolvimento'                , r"\benvolvimento\b", True),  # NEW — xlsx alt "Envolvimento"; GENERIC ('involvement') — false-positive risk
    ('engagement'              , 'NL'                      , 'betrokkenheid'               , r"\bbetrokkenheid\b", False),  # 0 funds — GENERIC ('involvement') — false-positive risk
    ('engagement'              , 'SV'                      , 'engagemang'                  , r"\bengagemang\b", False),  # 0 funds
    ('engagement'              , 'SV'                      , 'bolagsdialog'                , r"\bbolagsdialog\b", False),  # NEW — xlsx PRIMARY term for SV (corporate dialogue)
    ('engagement'              , 'SV'                      , 'paverkansarbete'             , r"\bp[åa]verkansarbete\b", False),  # NEW — xlsx alt "Påverkansarbete" (advocacy/impact work)
    ('engagement'              , 'SV'                      , 'paverkansdialog'             , r"\bp[åa]verkansdialog\b", False),  # NEW — xlsx alt "Påverkansdialog" (impact dialogue)
    ('engagement'              , 'NO'                      , 'engasjement'                 , r"\bengasjement\b", False),  # 0 funds
    ('engagement'              , 'NO'                      , 'eierengasjement'             , r"\beierengasjement\b", False),  # NEW — xlsx alt "Eierengasjement" (owner engagement)
    ('engagement'              , 'FI'                      , 'vaikuttaminen'               , r"\bvaikuttamin", False),  # 0 funds — GENERIC ('influencing') — false-positive risk

    # ===== ACTIVE OWNERSHIP ===== (NEW concept, from xlsx)
    ('active ownership'        , 'EN'                      , 'active_ownership'            , r"\bactive[\s\-]ownership\b", False),  # NEW
    ('active ownership'        , 'FR'                      , 'actionnariat_actif'          , r"\bactionnariat[\s\-]actif\b", False),  # NEW
    ('active ownership'        , 'ES'                      , 'activismo_accionarial'       , r"\bactivismo[\s\-]accionarial\b", False),  # NEW
    ('active ownership'        , 'ES'                      , 'propiedad_activa'            , r"\bpropiedad[\s\-]activa\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'IT'                      , 'azionariato_attivo'          , r"\bazionariato[\s\-]attivo\b", False),  # NEW
    ('active ownership'        , 'PT'                      , 'azionariato_attivo_pt'       , r"\bazionariato[\s\-]attivo\b", False),  # NEW — xlsx lists this literally under Portuguese too (looks like it may just mirror the Italian term); see 'propriedade_ativa' below for the more idiomatic PT alt
    ('active ownership'        , 'PT'                      , 'propriedade_ativa'           , r"\bpropriedade[\s\-]ativa\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'NL'                      , 'actief_aandeelhouderschap'   , r"\bactief[\s\-]aandeelhouderschap\b", False),  # NEW
    ('active ownership'        , 'NL'                      , 'actieve_eigenaarschap'       , r"\bactieve[\s\-]eigenaarschap\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'SV'                      , 'aktivt_agarskap'             , r"\baktivt[\s\-][äa]garskap\b", False),  # NEW
    ('active ownership'        , 'SV'                      , 'aktivt_agande'               , r"\baktivt[\s\-][äa]gande\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'SV'                      , 'agarengagemang'              , r"\b[äa]garengagemang\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'SV'                      , 'agarstyrning'                , r"\b[äa]garstyrning\b", False),  # NEW — xlsx alt
    ('active ownership'        , 'DA'                      , 'aktivt_ejerskab'             , r"\baktivt[\s\-]ejerskab\b", False),  # NEW
    ('active ownership'        , 'NO'                      , 'aktivt_eierskap'             , r"\baktivt[\s\-]eierskap\b", False),  # NEW
    ('active ownership'        , 'NO'                      , 'ansvarlig_eier_ao'           , r"\bansvarlig[\s\-]eier\b", False),  # NEW — xlsx alt "Ansvarlig eier"; also listed under Steward(s), added there too (harmless duplicate label)
    ('active ownership'        , 'FI'                      , 'aktiivinen_omistajuus'       , r"\baktiivinen[\s\-]omistajuus\b", False),  # NEW
    ('active ownership'        , 'FI'                      , 'omistaja_vaikuttaminen'      , r"\bomistaja[\s\-]vaikuttaminen\b", False),  # NEW — xlsx alt (two-word form)
    ('active ownership'        , 'FI'                      , 'omistajavaikuttaminen'       , r"\bomistajavaikuttaminen\b", False),  # NEW — xlsx alt (compound form)
 
    # ===== STEWARDSHIP =====
    ('stewardship'             , 'EN'                      , 'stewardship'                 , r"\bstewards?(?:hip)?\b", False),  # 2 funds — matches steward / stewards / stewardship
    ('stewardship'             , 'FR/DE/ES/IT/PT/DA/NO'    , 'stewardship_loanword'        , r"\bstewardship\b", False),  # 0 funds — English term used in these markets
    ('stewardship'             , 'FR'                      , 'actionnaire_responsable'     , r"\bactionnaire[\s\-]responsable\b", False),  # NEW — xlsx Steward(s) row, French
    ('stewardship'             , 'ES'                      , 'gestion_responsable'         , r"\bgesti[óo]n[\s\-]responsable\b", False),  # NEW — xlsx: Spanish native stewardship term
    ('stewardship'             , 'ES'                      , 'administracion_responsable'  , r"\badministraci[óo]n[\s\-]responsable\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'ES'                      , 'accionista_responsable'      , r"\baccionista[\s\-]responsable\b", False),  # NEW — xlsx Steward(s) row, Spanish
    ('stewardship'             , 'IT'                      , 'gestione_responsabile'       , r"\bgestione[\s\-]responsabile\b", False),  # NEW — xlsx: Italian native stewardship term
    ('stewardship'             , 'IT'                      , 'azionariato_responsabile'    , r"\bazionariato[\s\-]responsabile\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'IT'                      , 'azionista_responsabile'      , r"\bazionista[\s\-]responsabile\b", False),  # NEW — xlsx Steward(s) row, Italian
    ('stewardship'             , 'PT'                      , 'gestao_responsavel'          , r"\bgest[ãa]o[\s\-]respons[áa]vel\b", False),  # NEW — xlsx: Portuguese native stewardship term
    ('stewardship'             , 'PT'                      , 'propriedade_responsavel'     , r"\bpropriedade[\s\-]respons[áa]vel\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'PT'                      , 'acionista_responsavel'       , r"\bacionista[\s\-]respons[áa]vel\b", False),  # NEW — xlsx Steward(s) row, Portuguese
    ('stewardship'             , 'NL'                      , 'rentmeesterschap'            , r"\brentmeesterschap\b|\bstewardship\b", False),  # 0 funds
    ('stewardship'             , 'NL'                      , 'verantwoord_aandeelhouderschap', r"\bverantwoord[\s\-]aandeelhouderschap\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'NL'                      , 'verantwoord_aandeelhouder'   , r"\bverantwoord[\s\-]aandeelhouder\b", False),  # NEW — xlsx Steward(s) row, Dutch (singular)
    ('stewardship'             , 'SV'                      , 'forvaltarskap'               , r"\bf[öo]rvaltarskap\b|\bstewardship\b", False),  # 0 funds — forvaltarskap != forvaltning (=management)
    ('stewardship'             , 'SV'                      , 'ansvarsfullt_agande'         , r"\bansvarsfullt[\s\-][äa]gande\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'SV'                      , 'agaransvar'                  , r"\b[äa]garansvar\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'SV'                      , 'ansvarsfull_agare'           , r"\bansvarsfull[\s\-][äa]gare\b", False),  # NEW — xlsx Steward(s) row, Swedish
    ('stewardship'             , 'DA'                      , 'ansvarligt_ejerskab'         , r"\bansvarligt[\s\-]ejerskab\b", False),  # NEW — xlsx: Danish native stewardship term (DA had no native term before, only the loanword)
    ('stewardship'             , 'DA'                      , 'ansvarlig_ejer'              , r"\bansvarlig[\s\-]ejer\b", False),  # NEW — xlsx Steward(s) row, Danish
    ('stewardship'             , 'NO'                      , 'ansvarlig_eierskap'          , r"\bansvarlig[\s\-]eierskap\b", False),  # NEW — xlsx: Norwegian native stewardship term (NO had no native term before, only the loanword)
    ('stewardship'             , 'NO'                      , 'ansvarlig_eierstyring'       , r"\bansvarlig[\s\-]eierstyring\b", False),  # NEW — xlsx alt
    ('stewardship'             , 'NO'                      , 'ansvarlig_forvaltning'       , r"\bansvarlig[\s\-]forvaltning\b", False),  # NEW — xlsx alt; qualified by "ansvarlig" so kept False despite bare 'forvaltning' elsewhere meaning generic 'management'
    ('stewardship'             , 'NO'                      , 'ansvarlig_eier_steward'      , r"\bansvarlig[\s\-]eier\b", False),  # NEW — xlsx Steward(s) row, Norwegian; same phrase as active_ownership's alt (harmless duplicate label)
    ('stewardship'             , 'FI'                      , 'vastuullinen_omistajuus'     , r"\bvastuullinen[\s\-]omistajuus\b", False),  # NEW — xlsx: Finnish native stewardship term, distinct from 'omistajaohjaus' below
    ('stewardship'             , 'FI'                      , 'omistajaohjaus'              , r"\bomistajaohjaus\b|\bstewardship\b", False),  # 0 funds
    ('stewardship'             , 'FI'                      , 'vastuullinen_omistaja'       , r"\bvastuullinen[\s\-]omistaja\b", False),  # NEW — xlsx Steward(s) row, Finnish (singular)
]

COMPILED_PATTERNS = [
    (concept, languages, label, re.compile(pat, FLAGS), new)
    for concept, languages, label, pat, new in PATTERNS
]

n_concepts  = len({concept for concept, *_ in PATTERNS})
n_languages = len({lg for _, langs, *_ in PATTERNS for lg in langs.split("/")})
n_new       = sum(1 for *_, new in PATTERNS if new)
print(f"Loaded {len(COMPILED_PATTERNS)} patterns across "
      f"{n_concepts} concepts and {n_languages} language groups "
      f"({n_new} new from extractions).")


## CELL 3 — Abbreviation Expansion Map

Token-level expansion applied to matched fund names only. Best-effort: expands known tokens, flags residual unknowns in `Expansion_Complete` column, and records known-ambiguous/false-friend tokens (e.g. "Dev" = Developed vs Development, "CA" = Crédit Agricole brand vs anything else) in the `Flagged_Tokens` column instead of guessing a substitute for them.

In [ ]:
# Applied to matched tokens only — not full unabbreviation
# of the entire fund name (too error-prone at scale).
# ============================================================

# Token-level expansion: token -> expanded form.
# Lookup is case-insensitive (matched on stripped.lower()) so SUST / Sust /
# sust all resolve the same way; the dict below just holds the canonical
# output spelling. Merged from the multilingual list (see
# ABBREV_EXPANSION.docx) on top of the original English-only set.
#
# NOTE: "Dev" was removed from here — see KNOWN_NEGATIVE_ABBREVS below.
# The sustainable_development pattern already matches bare "dev" via regex
# (`dev(?:elopment)?`), so this loses no rescue.
ABBREV_EXPANSION = {
    # ── Shared / multi-language ──
    "Glb":      "Global",
    "Glbl":     "Global",
    "Imp":      "Impact",
    "Imptt":    "Impact",
    "Impct":    "Impact",
    "Enga":     "Engage",
    "Trans":    "Transition",
    "Trnstn":   "Transition",
    "Engmnt":   "Engagement",
    "Enggmnt":  "Engagement",
    "Pstv":     "Positive",
    "Scl":      "Social",
    "Aktn":     "Aktion",
    "Bdr":      "Bedre",
    "Omstil":   "Omstilling",
    "Hndlng":   "Handling",

    # ── English ──
    "Sust":     "Sustainable",
    "Sus":      "Sustainable",
    "Sst":      "Sustainable",
    "Gens":     "Generations",
    "Clmt":     "Climate",
    "Clim":     "Climate",
    "Env":      "Environmental",
    "Soc":      "Social",
    "Gov":      "Governance",
    "Eq":       "Equity",          # most common meaning
    "Pos":      "Positive",
    "Btr":      "Better",
    "bttr":     "Better",
    "Chng":     "Change",
    "Actn":     "Action",
    "Devpmt":   "Development",
    "Fds":      "Funds",
    "Fd":       "Fund",
    "Mkt":      "Market",
    "Mkts":     "Markets",
    "Intl":     "International",
    "Intern":   "International",
    "Emg":      "Emerging",
    "Em":       "Emerging",
    "Cnsrv":    "Conservative",
    "Eqs":      "Equities",

    # ── French ──
    "Drbl":     "Durable",
    "Générat":  "Générations",

    # ── German ──
    "Nachha":   "Nachhaltige",
    "Bssr":     "Bessere",
    "Szl":      "Soziale",
    "Wndl":     "Wandel",
    "Positiv":  "Positiver",
    "Entwick":  "Entwicklung",
    "Wnd":      "Wende",

    # ── Spanish ──
    "Sosten":   "Sostenible",
    "Generac":  "Generaciones",
    "Mjr":      "Mejor",
    "Implica":  "Implicación",
    "Cmb":      "Cambio",
    "Desarr":   "Desarrollo",
    "Transic":  "Transición",
    "Accn":     "Acción",

    # ── Italian ──
    "Sosteni":  "Sostenibile",
    "Generaz":  "Generazioni",
    "Mglr":     "Migliore",
    "Coinvol":  "Coinvolgimento",
    "Cambiam":  "Cambiamento",
    "Svlpp":    "Sviluppo",
    "Transiz":  "Transizione",
    "Azn":      "Azione",

    # ── Portuguese ──
    "Sustent":  "Sustentável",
    "Grçs":     "Gerações",
    "Mlhr":     "Melhor",
    "Mdnç":     "Mudança",
    "Desenvo":  "Desenvolvimento",

    # ── Dutch ──
    "Drzm":     "Duurzame",
    "Genera":   "Generaties",
    "Verande":  "Verandering",
    "Ontwikk":  "Ontwikkeling",
    "Positie":  "Positieve",

    # ── Swedish ──
    "Hllbr":    "Hållbar",
    "Bolagsd":  "Bolagsdialog",
    "Föränd":   "Förändring",
    "Utveck":   "Utveckling",
    "Omställ":  "Omställning",

    # ── Danish ──
    "Forand":   "Forandring",
    "Udvik":    "Udvikling",

    # ── Norwegian ──
    "Bærekra":  "Bærekraftig",
    "Generas":  "Generasjoner",
    "Ssl":      "Sosial",
    "Utvik":    "Utvikling",

    # ── Finnish ──
    "Ilmst":    "Ilmasto",
    "Sosiaal":  "Sosiaalinen",
    "Prmp":     "Parempi",
    "Vaikutt":  "Vaikuttaminen",
    "Mts":      "Muutos",
    "Myönte":   "Myönteinen",
    "Khtys":    "Kehitys",
    "Srtym":    "Siirtymä",
    "Sukupo":   "Sukupolvet",
    "Tmt":      "Toimet",
}
ABBREV_EXPANSION_CI = {k.lower(): v for k, v in ABBREV_EXPANSION.items()}

# Tokens that are KNOWN AMBIGUOUS / false-friend risks (see
# ABBREV_EXPANSION.docx). These are NOT substituted into the reconstructed
# name — several of the source values are annotations, not real words
# (e.g. "Ambiguous (England/Enhanced/Energy)", "Crédit Agricole (fund
# brand)"), and pasting those into Name_Expanded would corrupt it. Instead:
# the original token is left as-is, Expansion_Complete is forced to False,
# and the reason is recorded in the new Flagged_Tokens output column for a
# human to check. This is the "downgrade to review" behaviour the old
# comment described but never implemented.
KNOWN_NEGATIVE_ABBREVS = {
    "Transp":  "Transportation, not Transition",
    "TransP":  "Transportation, not Transition",
    "Dev":     "Ambiguous: Developed (markets) vs Development",
    "Eng":     "Ambiguous: England / Enhanced / Energy",
    "Owners":  "Founder/owner-led companies (thematic), not necessarily impact-related",
    "Act":     "French: Actions (shares), not Action/climate action",
    "CA":      "Crédit Agricole (fund brand), not an abbreviation",
    "Akt":     "Aktien/Aktier = shares (DE/SV/DA/NO), not Aktion/Action",
    "VM":      "VM Vermögens-Management GmbH (fund brand), not an abbreviation",
    "DD":      "DoubleDividend (fund brand), not an abbreviation",
    "BM":      "Unrelated code (share class / thematic)",
    "Bolag":   "Swedish: Bolag = Company, generic term, not impact-specific",
}
KNOWN_NEGATIVE_ABBREVS_CI = {k.lower(): v for k, v in KNOWN_NEGATIVE_ABBREVS.items()}

def expand_name(fund_name: str) -> tuple[str, bool, list[str]]:
    """
    Best-effort token expansion. Returns (expanded_name, fully_expanded, flagged_notes).
    fully_expanded=False if any token could not be confidently expanded OR
    matched a known-ambiguous token. flagged_notes lists "TOKEN: reason"
    for every known-ambiguous token found (see KNOWN_NEGATIVE_ABBREVS).
    """
    # Same separator set as Cell 4's tokenizer (whitespace, hyphen, en dash,
    # ampersand, slash, parens, plus) — capturing group keeps the separators
    # themselves in the output so the name can be reconstructed exactly.
    # A whitespace-only split let tokens like "Eq-Income" or "(Eq)" stay
    # glued to punctuation and slip past ABBREV_EXPANSION untouched.
    tokens = re.split(r'([\s\-–&/()+]+)', fund_name)
    expanded_tokens = []
    fully_expanded = True
    flagged_notes = []
    for token in tokens:
        stripped = token.strip()
        stripped_ci = stripped.lower()
        if stripped_ci in ABBREV_EXPANSION_CI:
            expanded_tokens.append(ABBREV_EXPANSION_CI[stripped_ci])
        elif stripped_ci in KNOWN_NEGATIVE_ABBREVS_CI:
            # Known-ambiguous: keep the original token, don't guess a
            # substitute, just flag it for a human.
            expanded_tokens.append(token)
            fully_expanded = False
            flagged_notes.append(f"{stripped}: {KNOWN_NEGATIVE_ABBREVS_CI[stripped_ci]}")
        # Unknown-token heuristic stays keyed off the ORIGINAL casing (not
        # stripped_ci): being all-caps is the actual signal that a token looks
        # like an acronym/abbreviation rather than an ordinary word. Folding
        # this to any-case would flag plain fund-name words ("Fund", "World",
        # "Bond", "Value", "Euro"...) as unknown abbreviations.
        elif re.match(r'^[A-Z]{2,5}$', stripped) and stripped not in {
            "EUR", "USD", "GBP", "CHF", "JPY", "SEK", "NOK", "DKK",  # currencies
            "ETF", "UCITS", "ELTIF",                                    # fund types
            "ESG", "ISR", "SDG", "ODS", "ODD",                         # already-known acronyms
            "ACC", "INC", "CAP", "DIS",                                 # share class
            "UK", "US", "EU", "EM", "EMU",                              # geography
            "AI", "IT", "IP",                                           # tech / share class
        }:
            # Token we don't know how to expand -> mark as incomplete
            expanded_tokens.append(token)
            fully_expanded = False
        else:
            expanded_tokens.append(token)
    return "".join(expanded_tokens), fully_expanded, flagged_notes


## CELL 4 — Token Scan (exploratory)

Exploratory. Prints uppercase abbreviations in the full dataset not in the known-safe set. Run once to check for gaps in `ABBREV_EXPANSION` before committing to the full matching pass.

In [ ]:
# ============================================================
# SAFE_TOKENS — Full Reference
# Tokens confirmed to have no impact-keyword meaning.
# Organized by category for maintainability.
# ============================================================

SAFE_TOKENS = {

    # ── Currencies ─────────────────────────────────────────
    "EUR",   # Euro
    "USD",   # US Dollar
    "GBP",   # British Pound
    "CHF",   # Swiss Franc
    "JPY",   # Japanese Yen
    "SEK",   # Swedish Krona
    "NOK",   # Norwegian Krone
    "DKK",   # Danish Krone
    "HKD",   # Hong Kong Dollar
    "PLN",   # Polish Zloty
    "CZK",   # Czech Koruna
    "HUF",   # Hungarian Forint

    # ── Fund legal structures ───────────────────────────────
    "ETF",   # Exchange-Traded Fund
    "UCITS", # Undertakings for Collective Investment in Transferable Securities
    "ELTIF", # European Long-Term Investment Fund
    "AIF",   # Alternative Investment Fund
    "SICAV", # Société d'Investissement à Capital Variable (open-ended investment company)
    "FCP",   # Fonds Commun de Placement (French contractual fund)
    "SICAF", # Société d'Investissement à Capital Fixe (closed-ended)
    "OEIC",  # Open-Ended Investment Company (UK equivalent of SICAV)
    "FGR",   # Fonds voor Gemene Rekening (Dutch pooled fund structure)

    # ── ESG / responsible investment labels ────────────────
    "ESG",   # Environmental, Social and Governance — now in PATTERNS
    "SRI",   # Socially Responsible Investment — confirmed in scope (Dirk, 2026-06-23); now in PATTERNS
    "ISR",   # Investissement Socialement Responsable (French SRI label) — in PATTERNS
    "SDG",   # Sustainable Development Goals — in PATTERNS
    "ODS",   # Objetivos de Desarrollo Sostenible (Spanish SDG equivalent)
    "ODD",   # Objectifs de Développement Durable (French SDG equivalent)
    "CSR",   # Corporate Social Responsibility

    # ── Share class / series identifiers ───────────────────
    # Single letters
    "A", "B", "C", "D", "E", "F", "G", "H", "I",
    "J", "K", "L", "M", "N", "P", "Q", "R", "S",
    "T", "V", "W", "X", "Y", "Z",
    # Two-letter share class codes
    "AC",    # Accumulating Class
    "AD",    # Accumulating/Distributing Class
    "BI",    # Nordea institutional share class
    "BP",    # Base Portfolio class (JOHCM, Nordea)
    "DBI",   # Degroof Beleggingsfonds Institutioneel — Belgian institutional tax class
    "DIS",   # Distributing
    "EF",    # Equity Fund (share class suffix, e.g. SWC EF)
    "FC",    # Feeder/Founders Class
    "GF",    # Growth Fund share class
    "IA",    # Institutional Accumulating
    "IC",    # Institutional Class / Income Class
    "II",    # Second institutional series
    "IM",    # Institutional Management share class  ← also used as brand prefix (AXA IM)
    "INC",   # Income
    "KL",    # Kundeklasse (Danish: customer/institutional portfolio class)
    "LC",    # Local Currency / share class
    "LD",    # Local Distributing (DWS convention)
    "MH",    # LBPAM/French institutional share class (Mandats)
    "NA",    # North America (geographic designation in fund names)
    "PC",    # Pension Class
    "RC",    # Retail Class (French/Belgian convention)
    "RDT",   # Revenus Définitivement Taxés (Belgian institutional tax regime, paired with DBI)
    "REI",   # Real Estate Income
    "RET",   # Retail / Return class
    "SI",    # Share class identifier
    "XC",    # Extended/Exchange Class
    # Three-letter series codes
    "ACC",   # Accumulating
    "CAP",   # Capitalising
    "ESR",   # Épargne Salariale Responsable (French employee savings share class, Amundi)
    "GIF",   # HSBC Global Investment Funds (platform/series)
    "GSF",   # Goldman Sachs Funds (sub-series)
    "III",   # Third fund series (e.g. BNP Paribas III)
    "ISF",   # Schroder International Selection Fund (series)
    "FAM",   # Fund platform identifier (thematic Luxembourg SICAV platform)
    "WW",    # Worldwide (Baillie Gifford WW fund range)
    # Four/five-letter series
    "INVF",  # Invesco Funds (series)
    "MDPS",  # Fund platform/vehicle identifier (paired with TOBAM)
    "AXAWF", # AXA World Funds (series)
    "BNPP",  # BNP Paribas (abbreviated brand, used as series prefix)
    "FTGF",  # Franklin Templeton Global Funds (series)
    "UBAM",  # Union Bancaire Privée AM (series)

    # ── Fund manager / brand codes ──────────────────────────
    "AB",    # AllianceBernstein
    "AAF",   # Aegon Asset Funds (platform)
    "AKL",   # SEBinvest AKL (SEB's Danish sub-brand)
    "AM",    # Asset Management (generic suffix)
    "AXA",   # AXA Investment Managers
    "AZ",    # Azimut (Italian asset manager)
    "BGF",   # BlackRock Global Funds
    "BHF",   # BHF Asset Servicing (German)
    "BL",    # Banque de Luxembourg Investments
    "CM",    # Crédit Mutuel (CM-AM = Crédit Mutuel Asset Management)
    "CPR",   # CPR Asset Management (Amundi subsidiary)
    "CT",    # Columbia Threadneedle Investments
    "DBI",   # Degroof Petercam Belgian institutional class (see share class above)
    "DNCA",  # DNCA Finance (French, Natixis subsidiary)
    "DNB",   # DNB Asset Management (Norwegian, Den Norske Bank)
    "DPAM",  # Degroof Petercam Asset Management
    "DWS",   # DWS Investment (Deutsche Bank AM)
    "ERSTE", # Erste Asset Management (Austrian)
    "FSSA",  # First Sentier Stewart Asia (formerly First State Stewart Asia)
    "GAM",   # GAM Investments (Swiss)
    "GS",    # Goldman Sachs Asset Management
    "HSBC",  # HSBC Asset Management
    "JPM",   # J.P. Morgan Asset Management
    "KBC",   # KBC Asset Management (Belgian)
    "KBI",   # KBI Global Investors (Irish, Amundi subsidiary)
    "LBPAM", # La Banque Postale Asset Management (French)
    "LCL",   # LCL (Crédit Lyonnais subsidiary, French retail bank)
    "LGT",   # LGT Capital Partners (Liechtenstein)
    "LLB",   # Liechtensteinische Landesbank
    "LO",    # Lombard Odier Investment Managers
    "MFS",   # MFS Investment Management (Massachusetts Financial Services)
    "MS",    # Morgan Stanley Investment Management
    "NT",    # Northern Trust Asset Management
    "ODIN",  # ODIN Forvaltning (Norwegian asset manager)
    "QI",    # Quantitative Investing (Robeco QI sub-brand)
    "RBC",   # Royal Bank of Canada / RBC Global Asset Management
    "SEB",   # Skandinaviska Enskilda Banken (Swedish)
    "SG",    # Société Générale Asset Management
    "SWC",   # Swiss Life Asset Managers
    "THEAM", # Theam S.A. (BNP Paribas ETF/passive unit)
    "TOBAM", # TOBAM Asset Management (To Be A Mirror, French quant)
    "UBS",   # UBS Asset Management
    "UFF",   # Union Financière de France
    "JSS",   # J. Safra Sarasin (Swiss private bank)
    "WF",    # French asset manager brand (WF Actions series)

    # ── Domicile / geography ────────────────────────────────
    "CHN",   # China
    "EU",    # European Union
    "EM",    # Emerging Markets
    "EMU",   # European Monetary Union (Eurozone)
    "ES",    # Spain / Spanish share class designation
    "IE",    # Ireland (domicile)
    "LU",    # Luxembourg (ISIN prefix / domicile)
    "LUX",   # Luxembourg (domicile, long form)
    "NL",    # Netherlands
    "UK",    # United Kingdom
    "US",    # United States
    "USA",   # United States of America

    # ── Strategy / style descriptors ───────────────────────
    "EQ",    # Equity (Goldman Sachs Dutch fund convention)
    "ETI",   # Entreprises de Taille Intermédiaire (French: mid-sized companies)
    "PME",   # Petites et Moyennes Entreprises (French: SMEs)
    "RE",    # Real Estate
    "SMID",  # Small-Mid Cap
    "STOCK", # Equities/stocks (Erste AM convention for equity funds)
    "DXL",   # Share class used in LUX-domiciled fund series
    "FGR",   # Fonds voor Gemene Rekening (Dutch: common account fund, legal structure)
    "AI",    # Artificial Intelligence (thematic) / share class
    "IP",    # Intellectual Property / share class
    "KID",   # Key Information Document (regulatory)
    "KIID",  # Key Investor Information Document (regulatory)
    "FI",    # Fondo de Inversión (Spanish: investment fund)

}

# SRI confirmed in scope by Dirk (2026-06-23).
# Added to PATTERNS as ("sri", "EN", r"\bSRI\b", False) and to SAFE_TOKENS above.

In [ ]:
# Run once on the full dataset to surface unknown abbreviations
# before finalising the ABBREV_EXPANSION map.
# ============================================================

print("Loading data...")
df = pd.read_excel(INPUT_FILE)
print(f"  {len(df)} funds, {len(df.columns)} columns")

all_tokens = []
for name in df[NAME_COL].dropna():
    tokens = re.split(r'[\s\-–&/()+]+', str(name))
    all_tokens.extend(t for t in tokens if t)

token_counts = Counter(all_tokens)

unknown_abbrevs = [
    (tok, count)
    for tok, count in token_counts.most_common(500)
    if re.match(r'^[A-Z]{2,5}$', tok)
    and tok not in SAFE_TOKENS
    and count >= 2
]

print(f"\n{'Token':<12} {'Count':>6}   (possible meaning)")
print("─" * 45)
for tok, count in unknown_abbrevs[:40]:
    known = ABBREV_EXPANSION.get(tok, "?")
    print(f"{tok:<12} {count:>6}   {known}")


Loading data...
  5680 funds, 133 columns

Token         Count   (possible meaning)
─────────────────────────────────────────────
BNP              66   ?
FF               19   ?
RV               16   ?
UB               12   ?
CI               11   ?
CR               10   ?


## CELL 5 — Matching Function

Core matching logic. Returns all pattern hits for a single fund name string.

In [ ]:
def match_fund(fund_name: str) -> list[dict]:
    """
    Returns a list of match records for a single fund name.
    One record per pattern matched (a name may match multiple patterns).
    """
    matches = []
    for concept, languages, label, compiled_re, new in COMPILED_PATTERNS:
        m = compiled_re.search(fund_name)
        if m:
            matches.append({
                "matched_concept":       concept,
                "matched_pattern_label": label,
                "matched_language":      languages,
                "matched_text":          m.group(0),
                # 5th pattern field is new_from_extraction; surfaced here as the
                # review flag so downstream summary/output cells work unchanged.
                "needs_review":          new,
            })
    return matches


## CELL 6 — Run Matching

Applies `match_fund()` across all funds. Attaches all objective and strategy text columns to each match row.

**Two passes per fund, for every fund:** (1) raw name; (2) abbreviation-expanded name (Cell 3), even if pass 1 already found something — this is deliberately recall-biased, so a fund can pick up extra pattern labels from its expanded name on top of confirmed raw hits. Anything found only in pass 2 is force-flagged `needs_review=True` and tagged `Matched_Via="expansion"`, since it rests on `expand_name()`'s best-effort guesses rather than the literal fund name.

In [ ]:
print("Running pattern matching...")

# Only keep objective columns that exist in this dataset
available_obj_cols = [c for c in OBJECTIVE_COLUMNS if c in df.columns]

results = []
n_expansion_only_hits = 0

for _, row in df.iterrows():
    fund_name = str(row.get(NAME_COL, ""))
    fund_id   = row.get(ID_COL, "")

    expanded_name, fully_expanded, flagged_notes = expand_name(fund_name)

    # Pass 1: raw name — unchanged behaviour, unchanged needs_review values.
    raw_matches = match_fund(fund_name)
    raw_labels  = {m["matched_pattern_label"] for m in raw_matches}
    combined = [{**m, "Matched_Via": "raw"} for m in raw_matches]

    # Pass 2: runs for EVERY fund, not only ones with zero raw hits. Biasing
    # toward recall on purpose: a false positive gets caught at review, a
    # false negative is never seen again. Any pattern label that only shows
    # up via the expanded name (not the raw name) is added and force-flagged
    # needs_review=True, since it rests on expand_name()'s best-effort
    # "most common meaning" guesses (see Cell 3) rather than the literal name.
    if expanded_name != fund_name:
        for m in match_fund(expanded_name):
            if m["matched_pattern_label"] not in raw_labels:
                combined.append({**m, "needs_review": True, "Matched_Via": "expansion"})
                n_expansion_only_hits += 1

    if not combined:
        continue

    # Collect all objective/strategy text for output columns
    obj_texts = {col: row.get(col, "") for col in available_obj_cols}

    # One row per matched pattern
    for match in combined:
        results.append({
            ID_COL:              fund_id,
            NAME_COL:            fund_name,
            "Name_Expanded":     expanded_name,
            "Expansion_Complete": fully_expanded,
            "Flagged_Tokens":    "; ".join(flagged_notes),
            **match,
            **obj_texts,
        })

results_df = pd.DataFrame(results)
print(f"  {results_df[ID_COL].nunique()} funds matched across {len(results_df)} pattern hits")
print(f"  of which {n_expansion_only_hits} pattern hits were found ONLY via abbreviation "
      f"expansion (not present on the raw name) — all flagged needs_review=True, "
      f"see Matched_Via column")


## CELL 7 — Summary Statistics

Printed console summary: total candidates, hit counts by pattern label and language group, review flag count.

In [ ]:
if len(results_df) == 0:
    print("No matches found.")
else:
    # Deduplicate to one row per fund for counting
    funds_df = results_df.drop_duplicates(subset=ID_COL)

    print(f"\n{'═'*55}")
    print(f"  IMPACT FUND CANDIDATES: {len(funds_df)} of {len(df)} funds")
    print(f"  ({len(funds_df)/len(df)*100:.1f}% of full dataset)")
    print(f"{'═'*55}")

    print(f"\nHits by pattern label:")
    for label, count in results_df["matched_pattern_label"].value_counts().items():
        n_review = results_df[results_df["matched_pattern_label"] == label]["needs_review"].sum()
        flag = "  ⚠ needs review" if n_review > 0 else ""
        print(f"  {label:<30} {count:>4}{flag}")

    print(f"\nHits by language group:")
    for lang, count in results_df["matched_language"].value_counts().items():
        print(f"  {lang:<10} {count:>4}")

    review_count = results_df["needs_review"].sum()
    print(f"\nRows flagged for human review: {review_count} "
          f"({review_count/len(results_df)*100:.1f}% of all hits)")

    if "Matched_Via" in results_df.columns:
        exp_hits  = (results_df["Matched_Via"] == "expansion").sum()
        exp_funds = results_df[results_df["Matched_Via"] == "expansion"][ID_COL].nunique()
        print(f"\nOf which, found ONLY via abbreviation expansion (not on the raw name): "
              f"{exp_hits} pattern hits across {exp_funds} funds")


## CELL 8 — Output to Excel

Writes four sheets: **Matches** (one row per hit), **Funds_Deduped** (one row per fund), **Summary** (pattern-level stats), **Token_Scan** (unknown abbreviation candidates).

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile = OUTPUT_DIR / f"Impact_Fund_Candidates_{timestamp}.xlsx"

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # Sheet 1 — Full results (one row per pattern hit)
    results_df.to_excel(writer, sheet_name="Matches", index=False)

    # Sheet 2 — One row per fund (first/most-significant match)
    if len(results_df) > 0:
        deduped = (
            results_df
            .sort_values("needs_review")          # confirmed hits first
            .drop_duplicates(subset=ID_COL, keep="first")
        )
        deduped.to_excel(writer, sheet_name="Funds_Deduped", index=False)

    # Sheet 3 — Summary statistics
    if len(results_df) > 0:
        summary_rows = []
        for label, grp in results_df.groupby("matched_pattern_label"):
            lang = grp["matched_language"].iloc[0]
            n_funds = grp[ID_COL].nunique()
            n_review = int(grp["needs_review"].sum())
            example = grp[NAME_COL].iloc[0]
            summary_rows.append({
                "Pattern Label":    label,
                "Language":         lang,
                "Funds Matched":    n_funds,
                "Needs Review (n)": n_review,
                "Example Name":     example,
            })
        summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)

    # Sheet 4 — Token scan: unknown abbreviations from Cell 4
    abbrev_rows = [
        {"Token": tok, "Count": count, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, count in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

print(f"\nOutput written to:\n  {outfile}")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_20260623_1120.xlsx


In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
outfile   = OUTPUT_DIR / f"Impact_Fund_Candidates_{timestamp}.xlsx"

# ── helpers ──────────────────────────────────────────────────────────────────

def _fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def _font(bold=False, size=10, color="000000"):
    return Font(name="Arial", bold=bold, size=size, color=color)

THIN = Border(
    left=Side(style="thin", color="D0D0D0"), right=Side(style="thin", color="D0D0D0"),
    top=Side(style="thin", color="D0D0D0"),  bottom=Side(style="thin", color="D0D0D0"),
)

def cell(ws, row, col, value="", bold=False, size=10, bg=None, fg="000000",
         halign="center", wrap=False):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = _font(bold, size, fg)
    c.alignment = Alignment(horizontal=halign, vertical="center", wrap_text=wrap)
    c.border    = THIN
    if bg:
        c.fill = _fill(bg)
    return c

def mwrite(ws, r1, c1, r2, c2, value="", bold=False, size=10, bg=None, fg="000000"):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    c = ws.cell(row=r1, column=c1, value=value)
    c.font      = _font(bold, size, fg)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border    = THIN
    if bg:
        c.fill = _fill(bg)
    return c

GREEN_D = "1A6B42"   # dark header
GREEN_L = "D6EFE3"   # section label row
GREEN_M = "EAF5ED"   # column headers inside sections
ALT_ROW = "F7FAF8"   # alternating row tint
BLUE_D  = "185FA5"   # Art 9
BLUE_L  = "DCE8F5"   # Art 8
AMBER   = "BA7517"   # needs review

# ── pre-compute all stats ─────────────────────────────────────────────────────

total_funds   = len(df)
matched_funds = results_df[ID_COL].nunique() if len(results_df) > 0 else 0
total_hits    = len(results_df)

if len(results_df) > 0:
    deduped = (
        results_df
        .sort_values("needs_review")
        .drop_duplicates(subset=ID_COL, keep="first")
    )
    n_review    = int(deduped["needs_review"].sum())
    n_confirmed = matched_funds - n_review

    # Summary table (existing)
    summary_rows = []
    for label, grp in results_df.groupby("matched_pattern_label"):
        summary_rows.append({
            "Pattern Label":    label,
            "Language":         grp["matched_language"].iloc[0],
            "Funds Matched":    grp[ID_COL].nunique(),
            "Needs Review (n)": int(grp["needs_review"].sum()),
            "Example Name":     grp[NAME_COL].iloc[0],
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("Funds Matched", ascending=False)

    # Language totals: unique funds matched by any pattern in that language
    lang_totals = (
        results_df.groupby("matched_language")[ID_COL]
        .nunique().sort_values(ascending=False)
    )

    # SFDR counts (join from full dataset)
    sfdr_col = "EU SFDR Fund type (Article 8 or Article 9)"
    if sfdr_col in df.columns:
        matched_ids = set(results_df[ID_COL])
        sfdr = df[df[ID_COL].isin(matched_ids)][sfdr_col].value_counts()
        n_art8 = int(sfdr.get("Article 8", 0))
        n_art9 = int(sfdr.get("Article 9", 0))
    else:
        n_art8 = n_art9 = 0

    # ── Term × Language long table ────────────────────────────────────────
    # Row = one (token, language, pattern_label) combination
    # Columns: token | language | pattern_label | # funds |
    #          % within language group | % of full dataset
    tl = (
        results_df
        .groupby(["matched_text", "matched_language", "matched_pattern_label"])[ID_COL]
        .nunique().reset_index().rename(columns={ID_COL: "n_funds"})
    )
    tl["pct_within_lang"] = tl.apply(
        lambda r: round(r["n_funds"] / lang_totals.get(r["matched_language"], 1) * 100, 1),
        axis=1,
    )
    tl["pct_of_dataset"] = (tl["n_funds"] / total_funds * 100).round(2)
    tl = tl.sort_values(["matched_language", "n_funds"], ascending=[True, False]).reset_index(drop=True)

    # ── Language matrices ─────────────────────────────────────────────────
    lang_order = list(lang_totals.index)

    pivot_c = (
        results_df.groupby(["matched_text", "matched_language"])[ID_COL]
        .nunique().unstack(fill_value=0).reset_index()
        .rename(columns={"matched_text": "Token"})
    )
    for lg in lang_order:                            # ensure all lang cols present
        if lg not in pivot_c.columns:
            pivot_c[lg] = 0
    pivot_c = pivot_c[["Token"] + lang_order]
    pivot_c["Total"] = pivot_c[lang_order].sum(axis=1)
    pivot_c = pivot_c.sort_values("Total", ascending=False).reset_index(drop=True)

    # rates version: each cell = n_funds / that language's total × 100
    pivot_r = pivot_c.copy()
    for lg in lang_order:
        lt = lang_totals.get(lg, 1)
        pivot_r[lg] = (pivot_r[lg] / lt * 100).round(1)
    pivot_r = pivot_r.rename(columns={"Total": "Total (funds)"})

    # lang-total footer rows to append to both pivots
    count_footer = {"Token": "— LANGUAGE TOTAL (unique funds) —"}
    rate_footer  = {"Token": "— LANGUAGE TOTAL (unique funds) —"}
    for lg in lang_order:
        count_footer[lg] = int(lang_totals.get(lg, 0))
        rate_footer[lg]  = int(lang_totals.get(lg, 0))
    count_footer["Total"]          = matched_funds
    rate_footer["Total (funds)"]   = matched_funds

else:
    deduped     = pd.DataFrame()
    n_review    = n_confirmed = 0
    summary_df  = pd.DataFrame()
    lang_totals = pd.Series(dtype=int)
    n_art8 = n_art9 = 0
    tl = pivot_c = pivot_r = pd.DataFrame()
    lang_order = []

# ── write workbook ────────────────────────────────────────────────────────────

with pd.ExcelWriter(outfile, engine="openpyxl") as writer:

    # ── existing sheets (unchanged) ───────────────────────────────────────
    results_df.to_excel(writer, sheet_name="Matches",      index=False)
    if len(deduped):
        deduped.to_excel(writer,    sheet_name="Funds_Deduped", index=False)
    if len(summary_df):
        summary_df.to_excel(writer, sheet_name="Summary",       index=False)

    abbrev_rows = [
        {"Token": tok, "Count": cnt, "Known Expansion": ABBREV_EXPANSION.get(tok, "UNKNOWN")}
        for tok, cnt in unknown_abbrevs[:100]
    ]
    pd.DataFrame(abbrev_rows).to_excel(writer, sheet_name="Token_Scan", index=False)

    if len(results_df) == 0:
        print("No matches — stats sheets skipped.")
    else:
        wb = writer.book

        # ── Sheet: Overview Stats ─────────────────────────────────────────
        ws = wb.create_sheet("Overview Stats")

        for i, w in enumerate([22, 18, 18, 18, 18, 18, 18, 18, 18, 18], 1):
            ws.column_dimensions[get_column_letter(i)].width = w

        r = 1
        mwrite(ws, r, 1, r, 9, "IMPACT FUND SCREENER — RUN OVERVIEW",
               bold=True, size=13, bg=GREEN_D, fg="FFFFFF")
        ws.row_dimensions[r].height = 28

        # Section: top-line counts (2 rows per box: label + value)
        r += 2
        mwrite(ws, r, 1, r, 9, "OVERVIEW", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        top_boxes = [
            ("Total funds\nin dataset",  f"{total_funds:,}"),
            ("Matched\nfunds",           f"{matched_funds:,}"),
            ("Match\nrate",              f"{matched_funds/total_funds*100:.1f}%"),
            ("Total\npattern hits",      f"{total_hits:,}"),
            ("Avg hits\nper fund",       f"{total_hits/max(matched_funds,1):.2f}"),
            ("Confirmed\nmatches",       f"{n_confirmed:,}"),
            ("Flagged for\nreview",      f"{n_review:,}"),
            ("Review\nrate",             f"{n_review/max(matched_funds,1)*100:.1f}%"),
        ]
        for ci, (lbl, val) in enumerate(top_boxes, 1):
            cell(ws, r,   ci, lbl, size=9,  bg=GREEN_M, fg="3A5A3A", wrap=True)
            cell(ws, r+1, ci, val, size=16, bg="FFFFFF", bold=True)
        ws.row_dimensions[r].height   = 30
        ws.row_dimensions[r+1].height = 32

        # Section: SFDR split
        r += 3
        mwrite(ws, r, 1, r, 9, "SFDR CLASSIFICATION", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        sfdr_boxes = [
            ("Article 8",         f"{n_art8:,}",                          BLUE_L,   BLUE_D),
            ("Article 9",         f"{n_art9:,}",                          BLUE_D,   "FFFFFF"),
            ("Article 9\nrate",   f"{n_art9/max(n_art8+n_art9,1)*100:.1f}%", "FFFFFF", "000000"),
        ]
        for ci, (lbl, val, bg, fg) in enumerate(sfdr_boxes, 1):
            cell(ws, r,   ci, lbl, size=9,  bg=bg, fg=fg, wrap=True)
            cell(ws, r+1, ci, val, size=16, bg=bg, fg=fg, bold=True)
        ws.row_dimensions[r].height   = 28
        ws.row_dimensions[r+1].height = 32

        # Section: language breakdown
        r += 3
        mwrite(ws, r, 1, r, 9, "MATCHES BY LANGUAGE GROUP", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        hdrs = ["Language", "Unique funds", "% of matched", "% of full dataset", "Top term", "Top term n"]
        for ci, h in enumerate(hdrs, 1):
            cell(ws, r, ci, h, bold=True, size=9, bg=GREEN_M, fg="3A5A3A")
        ws.row_dimensions[r].height = 16
        r += 1

        for i, (lang, lt) in enumerate(lang_totals.items()):
            sub = tl[tl["matched_language"] == lang]
            top = sub.sort_values("n_funds", ascending=False).iloc[0] if len(sub) else None
            bg_r = ALT_ROW if i % 2 else "FFFFFF"
            vals = [lang, lt, f"{lt/matched_funds*100:.1f}%",
                    f"{lt/total_funds*100:.2f}%",
                    top["matched_text"] if top is not None else "—",
                    int(top["n_funds"]) if top is not None else 0]
            for ci, v in enumerate(vals, 1):
                cell(ws, r, ci, v, size=10, bg=bg_r,
                     halign="left" if ci == 1 else "center")
            ws.row_dimensions[r].height = 15
            r += 1

        # Section: top 25 patterns
        r += 1
        mwrite(ws, r, 1, r, 9, "TOP 25 PATTERNS (by fund count)", bold=True, size=10, bg=GREEN_L, fg=GREEN_D)
        ws.row_dimensions[r].height = 18
        r += 1

        hdrs = ["Pattern label", "Language", "# funds", "% of matched", "% of full dataset",
                "Needs review n", "Needs review %"]
        for ci, h in enumerate(hdrs, 1):
            cell(ws, r, ci, h, bold=True, size=9, bg=GREEN_M, fg="3A5A3A")
        ws.row_dimensions[r].height = 16
        r += 1

        for i, (_, rd) in enumerate(summary_df.head(25).iterrows()):
            n    = rd["Funds Matched"]
            nrev = rd["Needs Review (n)"]
            bg_r = ALT_ROW if i % 2 else "FFFFFF"
            vals = [rd["Pattern Label"], rd["Language"], n,
                    f"{n/matched_funds*100:.1f}%",
                    f"{n/total_funds*100:.2f}%",
                    nrev,
                    f"{nrev/max(n,1)*100:.0f}%"]
            for ci, v in enumerate(vals, 1):
                cell(ws, r, ci, v, size=10, bg=bg_r,
                     halign="left" if ci <= 2 else "center")
            ws.row_dimensions[r].height = 15
            r += 1

        ws.freeze_panes = "A3"

        # ── Sheet: Term × Language (long format) ─────────────────────────
        ws2 = wb.create_sheet("Term × Language")

        tl_out = tl.rename(columns={
            "matched_text":            "Token",
            "matched_language":        "Language",
            "matched_pattern_label":   "Pattern Label",
            "n_funds":                 "# Funds",
            "pct_within_lang":         "% within language group",
            "pct_of_dataset":          "% of full dataset",
        })

        col_widths2 = [18, 12, 28, 10, 24, 18]
        for i, w in enumerate(col_widths2, 1):
            ws2.column_dimensions[get_column_letter(i)].width = w

        for ci, h in enumerate(tl_out.columns, 1):
            cell(ws2, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws2.row_dimensions[1].height = 18

        for ri, row_data in enumerate(tl_out.itertuples(index=False), start=2):
            bg_r = ALT_ROW if ri % 2 else "FFFFFF"
            for ci, v in enumerate(row_data, 1):
                cell(ws2, ri, ci, v, size=10, bg=bg_r,
                     halign="left" if ci <= 3 else "center")
            ws2.row_dimensions[ri].height = 14

        ws2.freeze_panes = "A2"

        # ── Sheet: Language Matrix — counts ──────────────────────────────
        ws3 = wb.create_sheet("Language Matrix (counts)")

        pivot_c_out = pd.concat([pivot_c, pd.DataFrame([count_footer])], ignore_index=True)
        pivot_c_out.columns = [str(c) for c in pivot_c_out.columns]

        ws3.column_dimensions["A"].width = 20
        for i in range(2, len(pivot_c_out.columns) + 1):
            ws3.column_dimensions[get_column_letter(i)].width = 12

        # header
        for ci, h in enumerate(pivot_c_out.columns, 1):
            cell(ws3, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws3.row_dimensions[1].height = 18

        # footer row index (last row)
        footer_row = len(pivot_c_out) + 1

        for ri, row_data in enumerate(pivot_c_out.itertuples(index=False), start=2):
            is_footer = (ri == footer_row)
            bg_r = GREEN_L if is_footer else (ALT_ROW if ri % 2 else "FFFFFF")
            for ci, v in enumerate(row_data, 1):
                cell(ws3, ri, ci, v, size=10, bg=bg_r, bold=is_footer,
                     halign="left" if ci == 1 else "center")
            ws3.row_dimensions[ri].height = 14

        ws3.freeze_panes = "B2"

        # ── Sheet: Language Matrix — rates (% within language group) ─────
        ws4 = wb.create_sheet("Language Matrix (rates %)")

        pivot_r_out = pd.concat([pivot_r, pd.DataFrame([rate_footer])], ignore_index=True)
        pivot_r_out.columns = [str(c) for c in pivot_r_out.columns]

        ws4.column_dimensions["A"].width = 20
        for i in range(2, len(pivot_r_out.columns) + 1):
            ws4.column_dimensions[get_column_letter(i)].width = 14

        for ci, h in enumerate(pivot_r_out.columns, 1):
            cell(ws4, 1, ci, h, bold=True, size=9, bg=GREEN_D, fg="FFFFFF")
        ws4.row_dimensions[1].height = 18

        note_col = len(pivot_r_out.columns) + 2
        ws4.cell(row=1, column=note_col, value="Note: % = funds matched by this token ÷ total funds matched by any pattern in that language group")
        ws4.cell(row=1, column=note_col).font = Font(name="Arial", size=8, italic=True, color="888888")

        footer_row = len(pivot_r_out) + 1

        for ri, row_data in enumerate(pivot_r_out.itertuples(index=False), start=2):
            is_footer = (ri == footer_row)
            bg_r = GREEN_L if is_footer else (ALT_ROW if ri % 2 else "FFFFFF")
            for ci, v in enumerate(row_data, 1):
                cell(ws4, ri, ci, v, size=10, bg=bg_r, bold=is_footer,
                     halign="left" if ci == 1 else "center")
            ws4.row_dimensions[ri].height = 14

        ws4.freeze_panes = "B2"

print(f"\nOutput written to:\n  {outfile}")
print(f"\nSheets: Matches | Funds_Deduped | Summary | Token_Scan |")
print(f"        Overview Stats | Term × Language |")
print(f"        Language Matrix (counts) | Language Matrix (rates %)")



Output written to:
  /Users/dannyhogan/Desktop/Hogan_RA_Work/Impact_Fund_Candidates_20260623_1323.xlsx

Sheets: Matches | Funds_Deduped | Summary | Token_Scan |
        Overview Stats | Term × Language |
        Language Matrix (counts) | Language Matrix (rates %)
